# Notebook 03: Validation Design

**Purpose:** Define the temporal validation strategy for this project.

- Implement a reusable walk-forward fold generator.
- Visualise fold boundaries on the training date_id range.
- Confirm the Stage 2 holdout split and the Stage 3 walk-forward plan.

All folds must respect the arrow of time — no random K-Fold.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.data.load_data import load_train, load_labels

train  = load_train()
labels = load_labels()

all_date_ids = train['date_id'].sort_values().values
T = len(all_date_ids)
print(f'Total training days: {T}  (date_id {all_date_ids[0]} – {all_date_ids[-1]})')

## 1. Stage 2 holdout split (simple)

In [ ]:
HOLDOUT_FRACTION = 0.20
holdout_size = int(T * HOLDOUT_FRACTION)
train_size   = T - holdout_size

split_date_id = all_date_ids[train_size]

train_ids = all_date_ids[:train_size]
val_ids   = all_date_ids[train_size:]

print(f'Train : {len(train_ids)} days  (date_id 0 – {train_ids[-1]})')
print(f'Val   : {len(val_ids)} days  (date_id {val_ids[0]} – {val_ids[-1]})')
print(f'Split at date_id = {split_date_id}')

# Visualise
fig, ax = plt.subplots(figsize=(12, 1.5))
ax.barh(0, train_size,              left=0,          height=0.4, color='steelblue', label='Train')
ax.barh(0, holdout_size,            left=train_size,  height=0.4, color='coral',    label='Validation')
ax.set_xlim(0, T)
ax.set_yticks([])
ax.set_xlabel('date_id')
ax.set_title('Stage 2 — Simple holdout split (80/20)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 2. Walk-forward fold generator

In [ ]:
def walk_forward_folds(
    date_ids: np.ndarray,
    n_folds: int = 5,
    val_fraction: float = 0.10,
    gap: int = 0,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Generate expanding-window walk-forward folds.

    Args:
        date_ids    : Sorted integer day indices.
        n_folds     : Number of folds.
        val_fraction: Fraction of total data used for each validation window.
        gap         : Gap (purge) between train end and val start in periods.

    Returns:
        List of (train_ids, val_ids) tuples, ordered by fold.
    """
    T = len(date_ids)
    val_size = max(1, int(T * val_fraction))
    folds = []

    for k in range(n_folds):
        val_end   = T - k * val_size
        val_start = val_end - val_size
        train_end = val_start - gap

        if train_end <= 0:
            break  # Not enough training data for this fold

        folds.append((
            date_ids[:train_end],
            date_ids[val_start:val_end],
        ))

    return list(reversed(folds))  # Chronological order


folds = walk_forward_folds(all_date_ids, n_folds=5, val_fraction=0.10)

print('Walk-forward fold summary:')
for i, (tr, vl) in enumerate(folds):
    print(f'  Fold {i+1}: train {tr[0]}–{tr[-1]} ({len(tr)} days)  '
          f'val {vl[0]}–{vl[-1]} ({len(vl)} days)')

In [ ]:
# Visualise walk-forward folds
colors = ['steelblue', 'coral']
fig, axes = plt.subplots(len(folds), 1, figsize=(12, len(folds) * 0.8), sharex=True)

for i, (tr, vl) in enumerate(folds):
    ax = axes[i]
    ax.barh(0, len(tr),         left=tr[0],  height=0.5, color='steelblue', alpha=0.8)
    ax.barh(0, len(vl),         left=vl[0],  height=0.5, color='coral',     alpha=0.9)
    ax.set_xlim(0, T)
    ax.set_yticks([])
    ax.set_ylabel(f'Fold {i+1}', rotation=0, labelpad=35, va='center', fontsize=8)

axes[-1].set_xlabel('date_id')
patch_train = mpatches.Patch(color='steelblue', label='Train')
patch_val   = mpatches.Patch(color='coral',     label='Validation')
fig.legend(handles=[patch_train, patch_val], loc='upper right', ncol=2)
fig.suptitle('Stage 3 — Walk-forward expanding window (5 folds)', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Gap/embargo sizing

For Stage 2 features (simple lags ≤ 10 days), a gap of 0–10 days is sufficient.
For Stage 3 features with rolling windows (e.g. 20-day rolling mean), increase the gap accordingly.

In [ ]:
# Show effect of purge gap on available training data
print('Effect of purge gap on fold 5 (longest training fold):')
for gap in [0, 5, 10, 20]:
    folds_gap = walk_forward_folds(all_date_ids, n_folds=5, val_fraction=0.10, gap=gap)
    if folds_gap:
        tr, vl = folds_gap[-1]
        print(f'  gap={gap:2d}: train={len(tr)} days, val={len(vl)} days')

## 4. Validation plan summary

| Stage | Method | Folds | Gap | Primary metric |
|-------|--------|-------|-----|----------------|
| 2 | Holdout 80/20 | 1 | 0 | Spearman-Sharpe |
| 3 | Walk-forward expanding | 5 | 0–10 days | Spearman-Sharpe |
| 4 | Walk-forward with purge | 5 | max feature lookback | Spearman-Sharpe |

The `walk_forward_folds()` function defined in this notebook should be moved to
`src/validation/folds.py` before Stage 3 work begins.